# Preprocessing of the Data. Download, import and get a first overview at the Data.

## New workflow: Decision-tree dataset for Neukoelln

This section is a compact, reproducible workflow:
1. Load all relevant shapefiles.
2. Clip geometries to Neukoelln.
3. Keep only the most important features for the decision tree.
4. Save as CSV (ML) and GPKG/SHP (geospatial).

### Step 1: Load and clip green roof data
These cells load the green roof and district layers, clean geometries, align CRS, and clip roofs to Neukoelln.

In [14]:
# Import required libraries
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [15]:
# Detect project root (notebook may run from /notebooks)
project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

# Build input paths based on the current folder structure
# Use GPKG instead of SHP to preserve full column names (SHP has 10-char limit)
roofs_path = project_root / "data" / "exports" / "preprocessing_step0_gr_roof_cl" / "buildings_cleaned_no_tiefgarage.gpkg"
neukoelln_path = project_root / "data" / "Bezirk_Neukoelln" / "Bezirk_Neukoelln.shp"

# Validate required input files
if not roofs_path.exists() or not neukoelln_path.exists():
    raise FileNotFoundError(
        f"Missing file.\nRoofs: {roofs_path}\nNeukoelln: {neukoelln_path}"
    )

In [16]:
def clean_geometries(gdf: gpd.GeoDataFrame, name: str) -> gpd.GeoDataFrame:
    """Repair invalid geometries when possible and drop unrecoverable ones."""
    start = len(gdf)

    # Remove empty or missing geometries first
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

    invalid_mask = ~gdf.is_valid
    if invalid_mask.any():
        invalid_count = int(invalid_mask.sum())
        try:
            gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].make_valid()
        except Exception:
            gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].buffer(0)
    else:
        invalid_count = 0

    # Drop geometries that are still invalid after repair
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty & gdf.is_valid].copy()
    removed = start - len(gdf)

    print(f"{name}: start={start}, invalid_before={invalid_count}, removed={removed}, remaining={len(gdf)}")
    return gdf

In [17]:
# Load shapefiles
green_roofs = gpd.read_file(roofs_path)
neukoelln = gpd.read_file(neukoelln_path)

# Clean geometries
green_roofs = clean_geometries(green_roofs, "Green roofs total")
neukoelln = clean_geometries(neukoelln, "District Neukoelln")

# Align CRS
if green_roofs.crs != neukoelln.crs:
    green_roofs = green_roofs.to_crs(neukoelln.crs)

# Clip roofs to Neukoelln boundary
green_roofs_nk = gpd.clip(green_roofs, neukoelln)
green_roofs_nk = clean_geometries(green_roofs_nk, "Green roofs Neukoelln")

# Print quick summary
print(f"Roofs file: {roofs_path}")
print(f"Neukoelln file: {neukoelln_path}")
print(f"Number of green roofs in Neukoelln: {len(green_roofs_nk)}")

Green roofs total: start=625479, invalid_before=0, removed=0, remaining=625479
District Neukoelln: start=1, invalid_before=0, removed=0, remaining=1
Green roofs Neukoelln: start=53349, invalid_before=0, removed=0, remaining=53349
Roofs file: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step0_gr_roof_cl\buildings_cleaned_no_tiefgarage.gpkg
Neukoelln file: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\Bezirk_Neukoelln\Bezirk_Neukoelln.shp
Number of green roofs in Neukoelln: 53349


### Step 2: Build model-ready feature table
These cells create the target label, compute roof area, select core numeric features, and clean the tabular dataset.

In [18]:
# Create target variable based on has_green_roof indicator (truncated to 'has_green_' due to SHP limit)
# 1 = has green roof, 0 = has no green roof
dataset_gdf = green_roofs_nk.copy()

# Check for the column (may be truncated to 'has_green_' due to Shapefile 10-char limit)
has_gr_col = "has_green_roof" if "has_green_roof" in dataset_gdf.columns else "has_green_"

if has_gr_col in dataset_gdf.columns:
    # Convert boolean to int (True=1, False=0)
    dataset_gdf["target_0_1"] = dataset_gdf[has_gr_col].astype(int)
    print(f"Target variable created from '{has_gr_col}' column")
    print(f"  - Class 1 (has green roof): {(dataset_gdf['target_0_1'] == 1).sum()}")
    print(f"  - Class 0 (no green roof): {(dataset_gdf['target_0_1'] == 0).sum()}")
else:
    raise KeyError("Column 'has_green_roof' or 'has_green_' not found. Ensure PreprocessingGreenRoofCleaning was run first.")

# Compute roof area as a robust core feature
area_calc = dataset_gdf
if area_calc.crs is not None and area_calc.crs.is_geographic:
    area_calc = area_calc.to_crs(25833)
dataset_gdf["roof_area_m2"] = area_calc.geometry.area.round(2)

Target variable created from 'has_green_roof' column
  - Class 1 (has green roof): 1377
  - Class 0 (no green roof): 51972


In [19]:
# Keep only key numeric feature columns if available
candidate_numeric_features = ["roof_area_m2", "gruen20_m2", "gruen20_p", "gint20_m2", "gex20_m2"]
feature_columns = [c for c in candidate_numeric_features if c in dataset_gdf.columns]

model_columns = feature_columns.copy()
if "target_0_1" in dataset_gdf.columns:
    model_columns = ["target_0_1"] + model_columns

# Validate that at least one model column exists
if not model_columns:
    raise ValueError("No suitable model columns found. Please verify column names.")

In [20]:
# Build model table without geometry
decision_tree_df = dataset_gdf.drop(columns=["geometry"], errors="ignore")[model_columns].copy()

# Clean numeric columns
for col in decision_tree_df.columns:
    if col != "target_0_1":
        decision_tree_df[col] = pd.to_numeric(decision_tree_df[col], errors="coerce")
        decision_tree_df[col] = decision_tree_df[col].fillna(decision_tree_df[col].median())

# Remove duplicate records
decision_tree_df = decision_tree_df.drop_duplicates()

# Print quick table summary
print("Final columns:", list(decision_tree_df.columns))
print("Dataset shape:", decision_tree_df.shape)
print(f"\nTarget variable distribution (target_0_1):")
print(decision_tree_df["target_0_1"].value_counts().sort_index())
print(f"  - Ratio class 1: {(decision_tree_df['target_0_1'] == 1).sum() / len(decision_tree_df) * 100:.2f}%")
decision_tree_df.head()

Final columns: ['target_0_1', 'roof_area_m2', 'gruen20_m2', 'gruen20_p', 'gint20_m2', 'gex20_m2']
Dataset shape: (21990, 6)

Target variable distribution (target_0_1):
target_0_1
0    20613
1     1377
Name: count, dtype: int64
  - Ratio class 1: 6.26%


,target_0_1,roof_area_m2,gruen20_m2,gruen20_p,gint20_m2,gex20_m2
19577,0,59.00,0.0,0.0,0.0,0.0
19575,0,20.00,0.0,0.0,0.0,0.0
19589,0,108.00,0.0,0.0,0.0,0.0
211502,0,57.88,0.0,0.0,0.0,0.0
19593,0,11.00,0.0,0.0,0.0,0.0


### Step 3: Export greenroof outputs
This cell exports the model table and geospatial layers to a dedicated subfolder.

In [21]:
# Export decision-tree outputs (CSV + GPKG + SHP) to a dedicated subfolder
output_dir = project_root / "data" / "exports" / "preprocessing_step1_clip_to_neuk"
output_dir.mkdir(parents=True, exist_ok=True)
output_csv = output_dir / "neukoelln_greenroofs.csv"
output_gpkg = output_dir / "neukoelln_greenroofs.gpkg"
output_shp = output_dir / "neukoelln_greenroofs.shp"

# Export tabular dataset
decision_tree_df.to_csv(output_csv, index=False)

# Keep selected model columns plus geometry for geospatial exports
export_gdf = dataset_gdf[[c for c in model_columns if c in dataset_gdf.columns] + ["geometry"]].copy()

# Export full layer to GPKG
export_gdf.to_file(
    output_gpkg,
    layer="decision_tree_dataset",
    driver="GPKG"
)

# Export only polygon geometries to SHP (no mixed geometry types)
polygon_types = {"Polygon", "MultiPolygon"}
export_gdf_poly = export_gdf[export_gdf.geometry.geom_type.isin(polygon_types)].copy()

if len(export_gdf_poly) > 0:
    export_gdf_poly.to_file(output_shp)
    print(f"SHP saved: {output_shp} ({len(export_gdf_poly)} features)")
else:
    print("No SHP exported: no polygon geometries found.")

# Print export summary
print(f"CSV saved: {output_csv}")
print(f"GPKG saved: {output_gpkg} ({len(export_gdf)} features)")

C:\Users\elbma\AppData\Local\Temp\ipykernel_12472\3975264951.py:26: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  export_gdf_poly.to_file(output_shp)
c:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'roof_area_m2' to 'roof_area_'
  ogr_write(


SHP saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step1_clip_to_neuk\neukoelln_greenroofs.shp (53337 features)
CSV saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step1_clip_to_neuk\neukoelln_greenroofs.csv
GPKG saved: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step1_clip_to_neuk\neukoelln_greenroofs.gpkg (53349 features)


### Step 4: Clip and export solar potential data
These cells clip the city-wide solar potential layer to Neukoelln and export SHP and GPKG outputs.

In [22]:
# Load city-wide solar potential layer
solar_total_path = project_root / "data" / "solarpotential_berlin" / "solarpotential.shp"
if not solar_total_path.exists():
    raise FileNotFoundError(f"Missing file: {solar_total_path}")

solar_total = gpd.read_file(solar_total_path)

#### Step 4.1: Process and clip data
Clean geometries, align CRS, clip to Neukoelln, and prepare polygon-only data for SHP export.

In [23]:
# Define clean_geometries locally if it is not available in the current kernel
if "clean_geometries" not in globals():
    def clean_geometries(gdf: gpd.GeoDataFrame, name: str) -> gpd.GeoDataFrame:
        start = len(gdf)
        gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

        invalid_mask = ~gdf.is_valid
        if invalid_mask.any():
            invalid_count = int(invalid_mask.sum())
            try:
                gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].make_valid()
            except Exception:
                gdf.loc[invalid_mask, "geometry"] = gdf.loc[invalid_mask, "geometry"].buffer(0)
        else:
            invalid_count = 0

        gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty & gdf.is_valid].copy()
        removed = start - len(gdf)
        print(f"{name}: start={start}, invalid_before={invalid_count}, removed={removed}, remaining={len(gdf)}")
        return gdf

# Clean and clip solar potential data
solar_total = clean_geometries(solar_total, "Solar potential total")
neukoelln = clean_geometries(neukoelln, "Bezirk Neukoelln")

if solar_total.crs != neukoelln.crs:
    solar_total = solar_total.to_crs(neukoelln.crs)

solar_nk = gpd.clip(solar_total, neukoelln)
solar_nk = clean_geometries(solar_nk, "Solar potential Neukoelln")

# Keep only polygon geometry for SHP export
polygon_types = {"Polygon", "MultiPolygon"}
solar_nk_poly = solar_nk[solar_nk.geometry.geom_type.isin(polygon_types)].copy()

Solar potential total: start=529946, invalid_before=0, removed=0, remaining=529946
Bezirk Neukoelln: start=1, invalid_before=0, removed=0, remaining=1
Solar potential Neukoelln: start=41975, invalid_before=0, removed=0, remaining=41975


#### Step 4.2: Export outputs of the solarpotential
Write the clipped solar potential data to SHP and GPKG in the preprocessing2 export folder.

In [24]:
# Export clipped layers to the dedicated subfolder
output_dir = project_root / "data" / "exports" / "preprocessing_step1_clip_to_neuk"
output_dir.mkdir(parents=True, exist_ok=True)
solar_nk_out_shp = output_dir / "Solarpotential_Neukoelln.shp"
solar_nk_out_gpkg = output_dir / "Solarpotential_Neukoelln.gpkg"

solar_nk_poly.to_file(solar_nk_out_shp)
solar_nk.to_file(solar_nk_out_gpkg, layer="solarpotential_neukoelln", driver="GPKG")

# Print export summary
print(f"Solar total: {solar_total_path}")
print(f"Solar Neukoelln (Shapefile, polygons only): {solar_nk_out_shp} ({len(solar_nk_poly)} features)")
print(f"Solar Neukoelln (GPKG, all geometry types): {solar_nk_out_gpkg} ({len(solar_nk)} features)")

Solar total: C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\solarpotential_berlin\solarpotential.shp
Solar Neukoelln (Shapefile, polygons only): C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step1_clip_to_neuk\Solarpotential_Neukoelln.shp (41964 features)
Solar Neukoelln (GPKG, all geometry types): C:\Users\elbma\Nextcloud\1. Semester\AI in Human Water-Systems\berlin-green-roofs\data\exports\preprocessing_step1_clip_to_neuk\Solarpotential_Neukoelln.gpkg (41975 features)


## Solar potential clipped to Neukoelln